# 13c · Bounded iterative retrieval

## The engineering question
Can a second evidence-seeking step fill a gap without turning retrieval into an unbounded agent loop?

**Learning outcomes:** compare a single search with a bounded planner/search loop; retain per-step evidence; stop on repeat/empty query or explicit limits; inspect a question whose missing fact remains missing; count planner and final-answer calls separately.

The three development scenarios ask for Atlas warranty plus reset duration, Boreal power budget plus maintenance duration, and Atlas warranty plus its undocumented waterproof rating. Their evidence targets are explicitly derived from fictional source documents; they are additional instructional scenarios, not the approved frozen test set. Two facts require two distinct evidence units. An unsupported requested fact must not be invented just because one part is answerable.

The initial search retrieves two chunks. At most two subsequent planner calls produce one query each, and each query retrieves two more. The final context budget is 384 tokens for both modes. Retrieval itself uses the local exact baseline to isolate planning; Qdrant integration is already exercised in 12b. The model never receives tool permissions or an open-ended execution loop.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from vector_masterclass.lesson import LessonRun
from vector_masterclass.config import root

plt.rcParams.update(
    {"figure.figsize": (8, 4), "axes.spines.top": False, "axes.spines.right": False}
)
lesson = LessonRun("13c", "G", docker=False)
print("Kernel:", sys.executable)
print("Workspace:", root())

In [ ]:
from transformers import AutoTokenizer
from vector_masterclass.config import model_path
from vector_masterclass.data import documents
from vector_masterclass.chunking import chunk_document
from vector_masterclass.models import embed, generate
from vector_masterclass.retrieval import exact_search
from vector_masterclass.context import assemble
from vector_masterclass.rag import answer, search_queries
from vector_masterclass.evidence import json_write

docs = documents(tenant="public")
chunks = [c for d in docs for c in chunk_document(d)]
lookup = {c.id: c for c in chunks}
vectors = embed([c.text for c in chunks])
tokenizer = AutoTokenizer.from_pretrained(str(model_path("generator")), local_files_only=True)
scenarios = [
    (
        "atlas-two-facts",
        "What is the Atlas warranty duration and how long must I hold its reset button?",
        {"atlas-warranty@1", "atlas-reset@1"},
    ),
    (
        "boreal-two-facts",
        "What is the Boreal PoE power budget and how long is its firmware maintenance window?",
        {"boreal-ports@1", "boreal-firmware@1"},
    ),
    (
        "missing-fact",
        "What is the Atlas warranty duration and its waterproof rating?",
        {"atlas-warranty@1", "UNSUPPORTED-waterproof-rating"},
    ),
]


def retrieve(text):
    return [
        lookup[id_]
        for id_, score in exact_search(embed([text], role="query")[0], vectors, list(lookup), k=2)
    ]


rows, traces = [], []
for id_, question, required in scenarios:
    initial = retrieve(question)
    accumulated = list(initial)
    seen_queries = {question.lower().strip()}
    steps = []
    stop_reason = "iteration limit"
    for iteration in range(2):
        current = assemble(accumulated, docs, tokenizer, 384, tenant="public")
        plan = generate(
            f"Question: {question}\nEvidence so far:\n{current['text']}\nWrite one short search query for the most important missing fact. If complete, output STOP. Do not answer the question.",
            max_new_tokens=64,
        )
        proposed = search_queries(plan["text"], limit=1)
        step = {"iteration": iteration, "planner": plan, "proposed": proposed}
        steps.append(step)
        if not proposed or proposed[0].strip().upper() == "STOP":
            stop_reason = "planner stopped or invalid output"
            break
        followup = proposed[0].lower().strip()
        if followup in seen_queries:
            stop_reason = "repeated query"
            break
        seen_queries.add(followup)
        found = retrieve(proposed[0])
        step["retrieved_ids"] = [c.id for c in found]
        accumulated.extend(found)
    for mode, selected in [("single pass", initial), ("bounded iteration", accumulated)]:
        context = assemble(selected, docs, tokenizer, 384, tenant="public")
        result = answer(question, context)
        present = {c.document_key for c in context["chunks"]}
        coverage = len(present & required) / len(required)
        rows.append(
            {
                "scenario": id_,
                "mode": mode,
                "required_source_coverage": coverage,
                "context_tokens": context["tokens"],
                "planner_calls": len(steps) if mode == "bounded iteration" else 0,
                "retrieval_calls": 1 + sum("retrieved_ids" in s for s in steps)
                if mode == "bounded iteration"
                else 1,
                "answer": result["text"],
                "citations_resolve": result["citation_audit"]["all_resolve"],
                "stop_reason": stop_reason
                if mode == "bounded iteration"
                else "single pass complete",
            }
        )
        traces.append(
            {
                "scenario": id_,
                "mode": mode,
                "steps": steps if mode == "bounded iteration" else [],
                "answer": result,
                "required_sources": sorted(required),
            }
        )
json_write(lesson.path / "iteration-traces.json", traces)
results = lesson.table(pd.DataFrame(rows))
display(results)
fig, ax = plt.subplots(figsize=(10, 4))
results.pivot(index="scenario", columns="mode", values="required_source_coverage").plot.bar(ax=ax)
ax.set(
    ylabel="Fraction of requested source targets present",
    ylim=(0, 1.05),
    title="A bounded loop cannot retrieve an absent fact",
)
ax.tick_params(axis="x", rotation=0)
ax.legend(loc="upper left", bbox_to_anchor=(1, 1))
fig.tight_layout()
lesson.figure(fig)
plt.show()

## Interpretation and exercise
The missing-fact scenario cannot reach full source coverage because no source supplies the waterproof rating. A good answer should qualify that part; inspect whether the model does so. More retrieval can also consume the fixed context budget with duplicates or distractors. A source-presence metric is coarser than the source-span coverage used earlier and does not establish answer correctness.

**Exercise:** inspect each step's proposed query and new source IDs. Identify a repeated or unhelpful query; if none occurred, cap the loop at one iteration and compare. Design a stopping rule that accounts for marginal evidence gain, token/time cost, and uncertainty without trusting the model's STOP output as proof of completeness.

Iterative RAG is an orchestration choice. Production budgets must include planner inference, embedding, search, reranking, context growth, and final generation. A larger iteration limit is not a correctness fix. An absent fact remains absent after every allowed iteration.

In [ ]:
manifest = lesson.finish(
    {
        "iteration_limits_hold": bool(
            results.planner_calls.le(2).all() and results.retrieval_calls.le(3).all()
        ),
        "same_context_budget": bool(results.context_tokens.le(384).all()),
        "all_final_answers_live": all(t["answer"]["live"] for t in traces),
        "missing_fact_stays_missing": bool(
            results.loc[results.scenario.eq("missing-fact"), "required_source_coverage"].lt(1).all()
        ),
        "baseline_comparison_present": len(results) == 6,
    },
    observations={
        "maximum_planner_calls_per_scenario": 2,
        "maximum_retrieval_calls_per_scenario": 3,
        "final_answer_calls": 6,
        "scenario_labels": "author-defined instructional source targets, separate from frozen gold queries",
    },
)

## References and next steps
Consult the [Qdrant documentation](https://qdrant.tech/documentation/),
the [MiniLM model card](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2),
and [uv's Jupyter guide](https://docs.astral.sh/uv/guides/integration/jupyter/).
The exact pinned configuration is in `configs/models.json`; worked exercises are in `docs/SOLUTIONS.md`.
This notebook's measurements apply to the recorded local run, not a production deployment.